# Notebook 4: Głębokie sieci neuronowe (Deep Learning)

## Cel
Implementacja i porównanie 3 architektur głębokich sieci neuronowych:
1. **CNN1D** – prosta sieć konwolucyjna 1D (3 bloki konwolucyjne + 2 FC)
2. **ResNet1D** – głęboka sieć rezydualna (He et al., 2016, adaptacja 1D)
3. **BiLSTM** – dwukierunkowa sieć LSTM z mechanizmem uwagi (attention)

Dane wejściowe: surowe przetworzone sygnały EKG (1000×12) → (12, 1000) dla Conv1d.
Ocena bez strojenia hiperparametrów (Kontrola 2).

**Wymaganie:** Kontrola pośrednia nr 2 – wszystkie architektury DL (bez optymalizacji)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay,
)

sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata, build_labels, load_raw_data,
    get_train_val_test_split, SUPERCLASSES,
)
from src.preprocessing import preprocess_batch
from src.deep_models import (
    ECGDataset, CNN1D, ResNet1D, BiLSTMClassifier,
    Trainer, get_device, get_deep_models,
)

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
FS = 100
DEVICE = get_device()

print(f"PyTorch {torch.__version__}")
print(f"Urządzenie obliczeniowe: {DEVICE}")


## 1. Przygotowanie danych

Wczytujemy przetworzone sygnały EKG i tworzymy PyTorch DataLoader'y.


In [ ]:
# Wczytaj metadane i etykiety
Y = load_ptbxl_metadata(DATA_PATH)
Y = build_labels(Y, DATA_PATH)
train_idx, val_idx, test_idx = get_train_val_test_split(Y)

# ── Podzbiór do demonstracji ────────────────────────────────────────────────
# Zmień N_PER_CLASS=None dla pełnego zbioru (trening ~1-2h na CPU)
N_PER_CLASS = 80   # 80 przykładów na klasę w treningu

def sample_per_class(df, idx, n=None):
    sampled = []
    for cls in SUPERCLASSES:
        cls_idx = df[(df.label_single == cls) & df.index.isin(idx)].index
        sampled.extend(cls_idx[:n] if n else cls_idx)
    return df.loc[sampled]

Y_train_s = sample_per_class(Y, train_idx, N_PER_CLASS)
Y_val_s   = sample_per_class(Y, val_idx,   N_PER_CLASS // 4)
Y_test_s  = sample_per_class(Y, test_idx,  N_PER_CLASS // 4)

print(f"Trening:   {len(Y_train_s):,} | Walidacja: {len(Y_val_s):,} | Test: {len(Y_test_s):,}")


In [ ]:
# Wczytaj surowe sygnały
print("Ładowanie i przetwarzanie sygnałów...")
def load_and_process(Y_df):
    X_raw = load_raw_data(Y_df, sampling_rate=FS, data_path=DATA_PATH)
    X_proc = preprocess_batch(X_raw, fs=FS, verbose=False)
    return X_proc

X_train = load_and_process(Y_train_s)
X_val   = load_and_process(Y_val_s)
X_test  = load_and_process(Y_test_s)

# Koduj etykiety
le = LabelEncoder()
le.fit(SUPERCLASSES)
y_train = le.transform(Y_train_s['label_single'].values)
y_val   = le.transform(Y_val_s['label_single'].values)
y_test  = le.transform(Y_test_s['label_single'].values)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Klasy: {le.classes_}")


In [ ]:
# Utwórz DataLoader'y PyTorch
BATCH_SIZE = 32

train_dataset = ECGDataset(X_train, y_train)
val_dataset   = ECGDataset(X_val,   y_val)
test_dataset  = ECGDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sprawdź kształt batcha
X_batch, y_batch = next(iter(train_loader))
print(f"Kształt batcha wejściowego: {X_batch.shape}")
print(f"  (batch={X_batch.shape[0]}, leads={X_batch.shape[1]}, samples={X_batch.shape[2]})")
print(f"Kształt etykiet: {y_batch.shape}")


## 2. Architektura 1: CNN1D

Prosta sieć konwolucyjna 1D złożona z 3 bloków konwolucyjnych,
z BatchNorm, ReLU i MaxPool po każdym bloku.


In [ ]:
# Inicjalizacja modelu CNN1D
cnn_model = CNN1D(num_channels=12, num_classes=len(SUPERCLASSES))

# Wyświetl architekturę
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN1D – liczba parametrów: {total_params:,}")
print()
print(cnn_model)


In [ ]:
# Trening CNN1D
print("\n=== Trening CNN1D ===")
EPOCHS = 20
LR = 1e-3

cnn_trainer = Trainer(cnn_model, device=DEVICE, learning_rate=LR)
cnn_history = cnn_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 3. Architektura 2: ResNet1D

Głęboka sieć rezydualna adaptowana dla sygnałów 1D. Połączenia skip-connection
(Skip connections / Residual connections) umożliwiają trening bardzo głębokich
sieci bez problemu zanikającego gradientu.


In [ ]:
# Inicjalizacja modelu ResNet1D
resnet_model = ResNet1D(num_channels=12, num_classes=len(SUPERCLASSES))

total_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"ResNet1D – liczba parametrów: {total_params:,}")
print()
print(resnet_model)


In [ ]:
# Trening ResNet1D
print("\n=== Trening ResNet1D ===")
resnet_trainer = Trainer(resnet_model, device=DEVICE, learning_rate=LR)
resnet_history = resnet_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 4. Architektura 3: Bidirectional LSTM

Dwukierunkowa sieć LSTM z mechanizmem uwagi (attention). LSTM modeluje
długoterminowe zależności w sygnale; mechanizm uwagi pozwala sieci skupić się
na najważniejszych momentach czasowych (np. kompleksach QRS).


In [ ]:
# Inicjalizacja modelu BiLSTM
lstm_model = BiLSTMClassifier(
    input_size=12, hidden_size=128,
    num_layers=2, num_classes=len(SUPERCLASSES)
)

total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"BiLSTM – liczba parametrów: {total_params:,}")
print()
print(lstm_model)


In [ ]:
# Trening BiLSTM
print("\n=== Trening BiLSTM ===")
lstm_trainer = Trainer(lstm_model, device=DEVICE, learning_rate=LR)
lstm_history = lstm_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 5. Krzywe uczenia (Loss i Accuracy)

In [ ]:
# Wizualizacja krzywych uczenia dla wszystkich 3 modeli
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_hist = {
    'CNN1D':    (cnn_history,    'steelblue'),
    'ResNet1D': (resnet_history, 'darkorange'),
    'BiLSTM':   (lstm_history,   'green'),
}

# Strata (Loss)
for name, (hist, color) in models_hist.items():
    axes[0].plot(hist['train_loss'], '--', color=color, alpha=0.7, label=f'{name} train')
    axes[0].plot(hist['val_loss'],   '-',  color=color, linewidth=2, label=f'{name} val')

axes[0].set_title('Krzywe straty (Loss)', fontweight='bold')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=8)

# Dokładność (Accuracy)
for name, (hist, color) in models_hist.items():
    axes[1].plot(hist['val_acc'], '-', color=color, linewidth=2, label=name)

axes[1].set_title('Dokładność walidacyjna (Val Accuracy)', fontweight='bold')
axes[1].set_xlabel('Epoka')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/deep_learning_training_curves.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Ewaluacja na zbiorze testowym

In [ ]:
# Ewaluacja wszystkich modeli na zbiorze testowym
results_dl = {}

for name, trainer in [
    ('CNN1D', cnn_trainer),
    ('ResNet1D', resnet_trainer),
    ('BiLSTM', lstm_trainer)
]:
    y_true, y_pred = trainer.predict(test_loader)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm  = confusion_matrix(y_true, y_pred)

    results_dl[name] = {'y_true': y_true, 'y_pred': y_pred,
                        'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w,
                        'confusion_matrix': cm}

    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"F1 Macro:    {f1m:.4f}")
    print(f"F1 Weighted: {f1w:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))


In [ ]:
# Macierze pomyłek DL
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, result) in zip(axes, results_dl.items()):
    disp = ConfusionMatrixDisplay(result['confusion_matrix'], display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={result["accuracy"]:.3f} | F1={result["f1_macro"]:.3f}',
                 fontweight='bold')

plt.tight_layout()
plt.savefig('../results/deep_learning_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Zbiorcze porównanie modeli DL
print("\n=== PORÓWNANIE ARCHITEKTUR DEEP LEARNING ===")
summary_dl = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': r['accuracy'],
        'F1 Macro': r['f1_macro'],
        'F1 Weighted': r['f1_weighted'],
        'Parametrów': sum(p.numel() for p in m.parameters() if p.requires_grad),
    }
    for (name, r), m in zip(
        results_dl.items(),
        [cnn_model, resnet_model, lstm_model]
    )
])
print(summary_dl.to_string(index=False))


In [ ]:
# Zapis modeli
os.makedirs('../models', exist_ok=True)

torch.save(cnn_model.state_dict(),    '../models/cnn1d.pt')
torch.save(resnet_model.state_dict(), '../models/resnet1d.pt')
torch.save(lstm_model.state_dict(),   '../models/bilstm.pt')

print("Zapisano modele:")
print("  models/cnn1d.pt")
print("  models/resnet1d.pt")
print("  models/bilstm.pt")


In [ ]:
print("\n=== PODSUMOWANIE – DEEP LEARNING ===")
print(f"\nZaimplementowane architektury:")
print(f"  1. CNN1D    – prosta sieć konwolucyjna 1D")
print(f"  2. ResNet1D – sieć rezydualna (skip connections)")
print(f"  3. BiLSTM   – dwukierunkowy LSTM z mechanizmem uwagi")
print(f"\nTrening: {EPOCHS} epok, LR={LR}, batch={BATCH_SIZE}")
print(f"Optymalizator: Adam + ReduceLROnPlateau")
print(f"Regularyzacja: Dropout, BatchNorm, Weight Decay, Gradient Clipping")
